# 9.Take the Institution name as input. Use Pydantic to define the schema for the desired output and create a custom output parser. Invoke the Chain and Fetch Results. Extract the below Institution related details from Wikipedia: The founder of the Institution. When it was founded. The current branches in the institution. How many employees are working in it. A brief 4-line summary of the institution.

In [20]:
import wikipediaapi
from pydantic import BaseModel
import re

In [21]:
class InstitutionDetails(BaseModel):
    name: str
    founder: str
    founded_year: str
    branches: str
    employees: str
    summary: str

In [22]:
def fetch_wikipedia_summary(institution_name):
    """
    Fetches the Wikipedia summary of an institution.
    """
    # Updated to include a user agent string as recommended by Wikipedia API
    wiki_wiki = wikipediaapi.Wikipedia(user_agent='ColabWikipediaApp/1.0 (your-email@example.com)', language='en')
    page = wiki_wiki.page(institution_name)
    if not page.exists():
        raise ValueError(f"Wikipedia page for '{institution_name}' not found.")
    return page.summary

In [27]:
def extract_institution_details(institution_name):
    """
    Extracts institution-related details from the Wikipedia summary.
    """
    summary = fetch_wikipedia_summary(institution_name)
    founder = re.search(r"Founded by (.*?)[.,]", summary)
    founded_year = re.search(r"Founded in (\d{4})", summary)
    branches = re.search(r"has (\d+) branches", summary)
    employees = re.search(r"employs approximately (\d+)", summary)

    details = InstitutionDetails(
        name=institution_name,
        founder=founder.group(1) if founder else "Not Found",
        founded_year=founded_year.group(1) if founded_year else "Not Found",
        branches=branches.group(1) if branches else "Not Found",
        employees=employees.group(1) if employees else "Not Found",
        summary=" ".join(summary.split(".")[:4])  # First 4 lines
    )
    return details

In [28]:
if __name__ == "__main__":
    institution_name = input("Enter the Institution Name: ")
    try:
        details = extract_institution_details(institution_name)
        print("\nExtracted Institution Details:")
        print(details.model_dump_json(indent=4))
    except Exception as e:
        print(f"Error: {e}")

Enter the Institution Name: MIT

Extracted Institution Details:
{
    "name": "MIT",
    "founder": "Not Found",
    "founded_year": "1861",
    "branches": "Not Found",
    "employees": "Not Found",
    "summary": "The Massachusetts Institute of Technology (MIT) is a private research university in Cambridge, Massachusetts, United States  Founded in 1861 to advance \"useful knowledge\", the university has played a significant role in the development of many areas of technology and science \nWilliam Barton Rogers founded MIT to accelerate American industrialization through scientific knowledge  Initially funded by a federal land grant, the institute adopted a German polytechnic model emphasizing laboratory instruction in applied science and engineering, and moved from Boston's Back Bay to its current campus in Cambridge in 1916"
}
